In [1]:
"""Kaggle GPU + Ollama + 带鉴权 OpenAI 兼容网关 + ngrok。
整段粘贴到一个 Python 单元运行；需要 GPU 和 Internet。
修复版：隔离 Host / Origin / Authorization，支持 Page Assist 跨域和 SSE。
持久化版：配合 Session options → Persistence = Files only 使用。
  - Kaggle 持久化恢复文件时会丢失可执行权限和符号链接，解压后的 Ollama 运行库无法直接复用。
  - 因此只持久化 Ollama 安装包（压缩文件）和模型；运行库每次开机解压到临时目录，不需重新下载。
  - ngrok 是单个二进制，恢复后补回可执行权限即可。
"""
import json
import hashlib
import os
from pathlib import Path
import re
import secrets
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.error
import urllib.request

# ==================== 用户配置 ====================
ROOT = Path("/kaggle/working/DeepSeek-Ollama")   # 必须位于 /kaggle/working 下才能被持久化
SOURCE_MODEL = "hf.co/mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF:Q4_K_M"
MODEL_NAME = "deepseek-v4-pro-qwen3.5-9b"
NUM_CTX = 8192             # 显存不足时改成 4096；先关闭占用显存的 ComfyUI 等服务
OLLAMA_PORT = 11434
PORT = 8188               # 保留原脚本的穿透目标端口
LOCAL_URL = f"http://127.0.0.1:{PORT}"
OLLAMA_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
# 可通过 Kaggle Secrets 设置 OPENAI_API_KEY；为空时自动生成并保存在 ROOT/api_key.txt。
# 必填：在 Kaggle Secrets 添加并授权 NGROK_AUTHTOKEN（不是 ngrok API Key）。
# 获取地址：https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok 免费账号自动提供开发域名，不需要自己的域名。

def get_secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name).strip()
    except Exception:
        return ""

def run_checked(*args, **kwargs):
    subprocess.run(list(args), check=True, **kwargs)

def log_tail(path):
    path = Path(path)
    if not path.exists():
        return "尚无日志"
    with path.open("rb") as f:
        f.seek(max(0, path.stat().st_size - 8000))
        return f.read().decode("utf-8", errors="replace")

def json_request(base, path, data=None, key=None, timeout=10):
    headers = {"Content-Type": "application/json", "ngrok-skip-browser-warning": "1"}
    if key:
        headers["Authorization"] = f"Bearer {key}"
    req = urllib.request.Request(base + path, headers=headers,
        data=None if data is None else json.dumps(data).encode())
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        detail = e.read(8000).decode("utf-8", errors="replace")
        raise RuntimeError(f"{path}: HTTP {e.code}: {detail}") from e

def ready(base, path, key=None):
    try:
        return json_request(base, path, key=key, timeout=2)
    except Exception:
        return None

def spawn(args, log_path, env=None):
    with Path(log_path).open("a", encoding="utf-8") as log:
        return subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT,
            env=env, start_new_session=True)

def wait_ready(process, base, path, log_path, key=None, seconds=120):
    deadline = time.monotonic() + seconds
    while not ready(base, path, key):
        if process.poll() is not None:
            raise RuntimeError("服务启动失败：\n" + log_tail(log_path))
        if time.monotonic() >= deadline:
            raise TimeoutError("等待服务超时：\n" + log_tail(log_path))
        time.sleep(1)

def download(source, target):
    target = Path(target)
    temporary = target.with_suffix(".download")
    for attempt in range(1, 4):
        try:
            req = urllib.request.Request(source, headers={"User-Agent": "Kaggle-Ollama-Setup"})
            with urllib.request.urlopen(req, timeout=120) as r, temporary.open("wb") as f:
                shutil.copyfileobj(r, f, 4 * 1024 * 1024)
                expected = r.headers.get("Content-Length")
            if expected and temporary.stat().st_size != int(expected):
                raise RuntimeError("下载大小校验失败")
            temporary.replace(target)
            return
        except Exception:
            if attempt == 3:
                raise
            print(f"下载中断，正在重试 {attempt}/3", flush=True)
            time.sleep(attempt * 2)

def make_executable(path):
    """Kaggle Persistence 恢复文件时会丢失 x 权限；对文件或目录下所有文件补回。
    返回实际被修复的文件数。"""
    path = Path(path)
    if path.is_dir():
        targets = [p for p in path.rglob("*") if p.is_file() and not p.is_symlink()]
    elif path.is_file():
        targets = [path]
    else:
        targets = []
    fixed = 0
    for target in targets:
        try:
            mode = target.stat().st_mode
            if mode & 0o111 != 0o111:
                target.chmod(mode | 0o755)
                fixed += 1
        except OSError:
            pass
    return fixed

ROOT.mkdir(parents=True, exist_ok=True)
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")
if not NGROK_AUTHTOKEN:
    raise RuntimeError("请先在 Kaggle Add-ons → Secrets 中添加并授权 NGROK_AUTHTOKEN。"
                       "获取地址：https://dashboard.ngrok.com/get-started/your-authtoken")
print("1/6 检查 GPU 与依赖...", flush=True)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("未检测到 NVIDIA GPU，请先在 Kaggle Settings 启用 GPU。")
print(gpu.stdout.strip(), flush=True)
run_checked(sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.115,<1",
    "uvicorn>=0.30,<1", "httpx>=0.27,<1", "zstandard>=0.23,<1", "pyngrok>=7.2,<10")

print("2/6 安装 / 启动 Ollama...", flush=True)
# 运行库放在不被持久化的临时目录：持久化会破坏其中的符号链接（如 libllama-common.so.0）
# 和可执行权限。持久化的只有安装包本身，开机时解压一次，约半分钟。
scratch = Path("/kaggle/tmp") if Path("/kaggle/tmp").is_dir() else Path("/tmp")
install = scratch / "ollama-runtime"
ollama = install / "bin/ollama"
archive = ROOT / "ollama-linux-amd64.tar.zst"
legacy_install = ROOT / "runtime"       # 旧版脚本解压在这里，恢复后已损坏

def stop_ollama():
    """停止本脚本的 Ollama 服务（含 llama-server 子进程），等待端口释放。"""
    process = globals().get("ollama_process")
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
    for pattern in (str(install), str(legacy_install)):
        subprocess.run(["pkill", "-f", pattern], capture_output=True)
    deadline = time.monotonic() + 20
    while ready(OLLAMA_URL, "/api/version") and time.monotonic() < deadline:
        time.sleep(0.5)

def extract_ollama():
    import zstandard
    shutil.rmtree(install, ignore_errors=True)
    install.mkdir(parents=True)
    with archive.open("rb") as f, zstandard.ZstdDecompressor().stream_reader(f) as reader:
        with tarfile.open(fileobj=reader, mode="r|") as tar:
            tar.extractall(install, filter="data")
    if not ollama.is_file():
        raise RuntimeError("Ollama 安装包结构不符，未找到 bin/ollama。")
    (install / ".install-complete").write_text("ok")

if not ollama.is_file() or not (install / ".install-complete").is_file():
    # 新会话或运行库缺失：先停掉可能仍从旧目录运行的服务，再准备运行库。
    stop_ollama()
    if archive.is_file():
        print("复用持久化的 Ollama 安装包，正在解压...", flush=True)
        try:
            extract_ollama()
        except Exception as e:
            print(f"安装包解压失败（{e}），重新下载...", flush=True)
            archive.unlink(missing_ok=True)
    if not archive.is_file():
        download("https://ollama.com/download/ollama-linux-amd64.tar.zst", archive)
        extract_ollama()
if legacy_install.exists():
    # 清理旧版留下的损坏运行库，释放持久化空间；不影响 models/。
    shutil.rmtree(legacy_install, ignore_errors=True)

env = os.environ.copy()
env.update(OLLAMA_HOST=f"127.0.0.1:{OLLAMA_PORT}",
    OLLAMA_MODELS=str(ROOT / "models"), OLLAMA_CONTEXT_LENGTH=str(NUM_CTX),
    OLLAMA_NUM_PARALLEL="1", OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_KEEP_ALIVE="30m",
    OLLAMA_NO_CLOUD="1")
if not ready(OLLAMA_URL, "/api/version"):
    ollama_log = ROOT / "ollama.log"
    ollama_process = spawn([str(ollama), "serve"], ollama_log, env)
    wait_ready(ollama_process, OLLAMA_URL, "/api/version", ollama_log)
print("Ollama", json_request(OLLAMA_URL, "/api/version"), flush=True)

print("3/6 拉取指定模型的 Q4_K_M 量化版并创建模型名...", flush=True)
models = json_request(OLLAMA_URL, "/api/tags").get("models", [])
if not any(m.get("name", "").lower() == SOURCE_MODEL.lower() for m in models):
    # Ollama 原生下载会校验 blob 摘要，并复用已有内容 / 未完成下载。
    run_checked(str(ollama), "pull", SOURCE_MODEL, env=env)
else:
    print("复用已下载的模型。", flush=True)
modelfile = ROOT / "Modelfile"
modelfile.write_text(f"FROM {SOURCE_MODEL}\nPARAMETER num_ctx {NUM_CTX}\n", encoding="utf-8")
# 继承 GGUF 导入时解析的模型模板；不手写可能破坏 tools/thinking 的通用模板。
run_checked(str(ollama), "create", MODEL_NAME, "-f", str(modelfile), env=env)
info = json_request(OLLAMA_URL, "/api/show", {"model": MODEL_NAME})
print("模型能力：", info.get("capabilities", "请查看 Ollama 日志"), flush=True)
print("正在进行本地推理自检，首次加载需要等待...", flush=True)
try:
    result = json_request(OLLAMA_URL, "/v1/chat/completions", {
        "model": MODEL_NAME, "messages": [{"role": "user", "content": "Reply with OK."}],
        "max_tokens": 32, "stream": False}, timeout=600)
    if not result.get("choices"):
        raise RuntimeError(f"推理返回格式异常：{result}")
except Exception as e:
    raise RuntimeError(f"模型加载 / 推理失败：{e}\n" + log_tail(ROOT / "ollama.log")) from e
loaded = json_request(OLLAMA_URL, "/api/ps").get("models", [])
active = next((m for m in loaded if m.get("name") in (MODEL_NAME, MODEL_NAME + ":latest")), None)
if active and active.get("size_vram") == 0:
    raise RuntimeError("模型落到纯 CPU，已停止发布。请查看 ollama.log 的 CUDA / 驱动错误，"
        "或把 NUM_CTX 改成 4096 后重试。")
print("本地推理通过；GPU 权重字节：", (active or {}).get("size_vram", "未知"), flush=True)

print("4/6 启动带 API Key 的 OpenAI 兼容网关...", flush=True)
key_path = ROOT / "api_key.txt"
API_KEY = get_secret("OPENAI_API_KEY")
if not API_KEY:
    API_KEY = key_path.read_text().strip() if key_path.exists() else "sk-" + secrets.token_urlsafe(32)
if len(API_KEY) < 16 or any(ch.isspace() for ch in API_KEY):
    raise ValueError("OPENAI_API_KEY 至少 16 个字符，且不能包含空白。")
key_path.write_text(API_KEY)
key_path.chmod(0o600)
TUNNEL_MODE = "ngrok"

GATEWAY_CODE = r'''
import hmac
import json
import os
from contextlib import asynccontextmanager
import httpx
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response, StreamingResponse
from starlette.background import BackgroundTask

API_KEY = os.environ["GATEWAY_API_KEY"]
UPSTREAM = os.environ["GATEWAY_UPSTREAM"]
HOP = {"host", "authorization", "connection", "keep-alive", "proxy-authenticate",
       "proxy-authorization", "te", "trailer", "transfer-encoding", "upgrade",
       "content-length", "content-encoding"}
FORWARD = {"content-type", "accept"}

@asynccontextmanager
async def lifespan(app):
    app.state.client = httpx.AsyncClient(timeout=httpx.Timeout(600, connect=10),
                                       trust_env=False)
    yield
    await app.state.client.aclose()

app = FastAPI(lifespan=lifespan, docs_url=None, redoc_url=None, openapi_url=None)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=False,
    allow_methods=["GET", "POST", "DELETE", "PUT", "PATCH", "OPTIONS"],
    allow_headers=["*"], expose_headers=["x-gateway-upstream-status"])

def error(message, status, kind="invalid_request_error"):
    return JSONResponse({"error": {"message": message, "type": kind,
                                  "param": None, "code": None}}, status_code=status)

@app.get("/healthz")
async def health():
    return {"service": "kaggle-ollama-gateway", "status": "ok"}

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "DELETE", "PUT", "PATCH"])
async def proxy(path: str, request: Request):
    auth = request.headers.get("authorization", "")
    scheme, _, token = auth.strip().partition(" ")
    token = token.strip()
    if scheme.lower() != "bearer" or not hmac.compare_digest(token.encode(), API_KEY.encode()):
        return error("Invalid API key", 401, "authentication_error")
    if not path or any(part in (".", "..") for part in path.split("/")):
        return error("Invalid API path", 400)
    body = await request.body()
    wants_stream = False
    if body:
        try:
            payload = json.loads(body)
            wants_stream = isinstance(payload, dict) and payload.get("stream") is True
        except (ValueError, UnicodeDecodeError):
            return error("Request body must be JSON", 400)
    # 整个 /v1 路由透传，保留 tools、response_format、reasoning 等请求字段。
    # 不公开 Ollama /api/pull、/api/create、/api/delete 等管理端点。
    target = f"{UPSTREAM}/v1/{path}"
    # 仅转发内容协商头。httpx 为 127.0.0.1 重建 Host；不把公网 Host、
    # 浏览器 Origin、Referer、Forwarded、Cookie 或网关密钥传给 Ollama。
    # 原版误写成 not in FORWARD，会触发 Ollama 的来源 / 主机校验 403。
    headers = {k: v for k, v in request.headers.items() if k.lower() in FORWARD}
    headers["accept-encoding"] = "identity"
    try:
        req = app.state.client.build_request(request.method, target,
            params=request.query_params.multi_items(), content=body, headers=headers)
        upstream = await app.state.client.send(req, stream=True)
    except httpx.HTTPError:
        return error("Ollama upstream unavailable or timed out", 502, "api_error")
    outgoing = {k: v for k, v in upstream.headers.items()
                if k.lower() not in HOP and not k.lower().startswith("access-control-")}
    outgoing["x-gateway-upstream-status"] = str(upstream.status_code)
    if wants_stream and upstream.status_code < 400:
        outgoing["cache-control"] = "no-cache"
        outgoing["x-accel-buffering"] = "no"
        async def relay():
            try:
                async for chunk in upstream.aiter_bytes():
                    yield chunk
            finally:
                await upstream.aclose()
        return StreamingResponse(relay(), status_code=upstream.status_code,
            headers=outgoing, background=BackgroundTask(upstream.aclose))
    try:
        content = await upstream.aread()
    except httpx.HTTPError:
        return error("Ollama response interrupted", 502, "api_error")
    finally:
        await upstream.aclose()
    return Response(content, status_code=upstream.status_code, headers=outgoing)
'''
gateway_path = ROOT / "gateway.py"
gateway_path.write_text(GATEWAY_CODE, encoding="utf-8")
signature = (GATEWAY_CODE, API_KEY, OLLAMA_URL, TUNNEL_MODE, PORT)
old_gateway = globals().get("gateway_process")
gateway_alive = old_gateway is not None and old_gateway.poll() is None
if gateway_alive and globals().get("_gateway_signature") != signature:
    old_gateway.terminate()
    old_gateway.wait(timeout=15)
    gateway_alive = False
if not gateway_alive:
    # 端口冲突给出明确错误，不误把原 ComfyUI 当成新 API 服务。
    import socket
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", PORT)) == 0:
            raise RuntimeError(f"端口 {PORT} 已被占用。请停止旧 ComfyUI，或修改顶部 PORT。")
    gateway_env = env.copy()
    gateway_env.update(GATEWAY_API_KEY=API_KEY, GATEWAY_UPSTREAM=OLLAMA_URL)
    gateway_log = ROOT / "gateway.log"
    gateway_process = spawn([sys.executable, "-m", "uvicorn", "gateway:app",
        "--app-dir", str(ROOT), "--host", "127.0.0.1", "--port", str(PORT),
        "--no-access-log"], gateway_log, gateway_env)
    _gateway_signature = signature
    wait_ready(gateway_process, LOCAL_URL, "/v1/models", gateway_log, API_KEY)
json_request(LOCAL_URL, "/v1/models", key=API_KEY)

# 模拟浏览器扩展访问：同时带公网 Host 和扩展 Origin，避免仅测 localhost 漏检。
BROWSER_ORIGIN = "chrome-extension://page-assist-self-test"
def check_browser_access(base, override_host=False):
    headers = {"Origin": BROWSER_ORIGIN, "ngrok-skip-browser-warning": "1"}
    if override_host:
        headers["Host"] = "page-assist-test.ngrok-free.app"
    preflight = urllib.request.Request(base + "/v1/models", method="OPTIONS", headers={
        **headers, "Access-Control-Request-Method": "GET",
        "Access-Control-Request-Headers": "authorization,content-type,ngrok-skip-browser-warning,x-client-info"})
    with urllib.request.urlopen(preflight, timeout=15) as r:
        if r.status != 200 or r.headers.get("Access-Control-Allow-Origin") not in ("*", BROWSER_ORIGIN):
            raise RuntimeError("浏览器 OPTIONS 预检失败")
    req = urllib.request.Request(base + "/v1/models", headers={
        **headers, "Authorization": "Bearer " + API_KEY})
    with urllib.request.urlopen(req, timeout=15) as r:
        if r.headers.get("Access-Control-Allow-Origin") not in ("*", BROWSER_ORIGIN):
            raise RuntimeError("实际响应缺少跨域许可")
        payload = json.load(r)
        if not any(m.get("id") in (MODEL_NAME, MODEL_NAME + ":latest") for m in payload.get("data", [])):
            raise RuntimeError("浏览器模型列表缺少目标模型")
    return True

try:
    check_browser_access(LOCAL_URL, override_host=True)
except Exception as e:
    raise RuntimeError("本地浏览器兼容性自检失败：" + str(e)) from e
print("本地网关：公网 Host / 扩展 Origin / OPTIONS 预检通过。", flush=True)

print("5/6 复用 / 创建 ngrok HTTPS 隧道...", flush=True)
from pyngrok import conf, ngrok

# 使用独立二进制和配置，只管理本脚本的 ngrok 进程。
# Authtoken 通过进程配置注入，不调用 set_auth_token 写入配置文件。
ngrok_binary = ROOT / "ngrok-runtime" / "ngrok"
ngrok_binary.parent.mkdir(exist_ok=True)
# 持久化恢复后 ngrok 二进制同样会丢失可执行权限。
make_executable(ngrok_binary)
ngrok_log = ROOT / "ngrok.log"

def redact_ngrok(value):
    return str(value).replace(NGROK_AUTHTOKEN, "[REDACTED]").replace(API_KEY, "[REDACTED]")

def on_ngrok_log(event):
    with ngrok_log.open("a", encoding="utf-8") as log:
        log.write(redact_ngrok(event) + "\n")

ngrok_signature = (LOCAL_URL, hashlib.sha256(NGROK_AUTHTOKEN.encode()).hexdigest())
previous_config = globals().get("_ngrok_config")
if previous_config is not None and globals().get("_ngrok_signature") != ngrok_signature:
    ngrok.kill(pyngrok_config=previous_config)

ngrok_config = conf.PyngrokConfig(
    ngrok_path=str(ngrok_binary), config_path=str(ROOT / "ngrok.yml"),
    auth_token=NGROK_AUTHTOKEN, ngrok_version="3", config_version="2",
    startup_timeout=60, request_timeout=15, start_new_session=True,
    log_event_callback=on_ngrok_log,
)
# 从上个 Cloudflare 版本升级时，只停止当前 Notebook 中记录的旧隧道。
old_cloudflare = globals().get("tunnel_process")
if old_cloudflare is not None and old_cloudflare.poll() is None:
    old_cloudflare.terminate()
    old_cloudflare.wait(timeout=15)

def open_tunnel():
    # get_tunnels 会按需安装和启动 ngrok；同一实例上重复运行会复用。
    tunnels = ngrok.get_tunnels(pyngrok_config=ngrok_config)
    tunnel = next((t for t in tunnels
        if t.public_url.startswith("https://")
        and str(t.config.get("addr", "")).rstrip("/") == LOCAL_URL), None)
    if tunnel is None:
        # 切换 PORT 时清理该专用实例的旧 endpoint，避免占用免费账号开发域名。
        for item in tunnels:
            ngrok.disconnect(item.public_url, pyngrok_config=ngrok_config)
        tunnel = ngrok.connect(addr=LOCAL_URL, proto="http", bind_tls=True,
            inspect=False, pyngrok_config=ngrok_config)
    return tunnel

try:
    try:
        ngrok_tunnel = open_tunnel()
    except Exception:
        # 恢复的 ngrok 二进制可能已损坏：删除后让 pyngrok 重新下载，再试一次。
        if not ngrok_binary.exists():
            raise
        try:
            ngrok.kill(pyngrok_config=ngrok_config)
        except Exception:
            pass
        ngrok_binary.unlink()
        print("恢复的 ngrok 无法启动，正在重新下载...", flush=True)
        ngrok_tunnel = open_tunnel()
    url = ngrok_tunnel.public_url.rstrip("/")
    if not url.startswith("https://"):
        raise RuntimeError("ngrok 未返回 HTTPS 地址。")
    _ngrok_config = ngrok_config
    _ngrok_signature = ngrok_signature
except Exception as e:
    # 不把包含 Authtoken 的异常 traceback 继续向外传播。
    detail = redact_ngrok(e)
    raise RuntimeError("ngrok 启动失败：" + detail +
        "\n请检查 NGROK_AUTHTOKEN、账号是否验证，以及免费额度 / 在线端点限制。"
        "\n日志：" + str(ngrok_log)) from None

print("6/6 验证公网模型列表和真实 SSE 响应...", flush=True)
public_ok = False
last_public_error = "尚无响应"
deadline = time.monotonic() + 90
while time.monotonic() < deadline:
    try:
        public = json_request(url, "/v1/models", key=API_KEY, timeout=10)
        if isinstance(public, dict) and any(m.get("id") in (MODEL_NAME, MODEL_NAME + ":latest")
                for m in public.get("data", [])):
            public_ok = True
            break
        last_public_error = "返回的模型列表不包含目标模型：" + str(public)[:500]
    except Exception as e:
        last_public_error = redact_ngrok(e)
    time.sleep(2)

def public_diagnostic():
    # ngrok 的 HTTP 边缘错误未必会出现在 agent 日志里，直接检查公网响应。
    req = urllib.request.Request(url + "/healthz", headers={"ngrok-skip-browser-warning": "1"})
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            return "公网 /healthz: HTTP " + str(r.status) + " " + r.read(2000).decode("utf-8", "replace")
    except urllib.error.HTTPError as e:
        body = e.read(6000).decode("utf-8", "replace")
        codes = sorted(set(re.findall(r"ERR_NGROK_\d+", body)))
        return f"公网 /healthz: HTTP {e.code}; ngrok 错误码: {codes}; 响应: {body[:2000]}"
    except Exception as e:
        return "公网 /healthz: " + str(e)

def check_stream(base):
    """验证真正的 SSE 帧、delta 和 [DONE]，不把 JSON 响应误报为流式成功。"""
    payload = {"model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Reply with OK."}],
        "max_tokens": 16, "stream": True}
    req = urllib.request.Request(base + "/v1/chat/completions",
        data=json.dumps(payload).encode(), headers={
            "Authorization": "Bearer " + API_KEY, "Content-Type": "application/json",
            "Accept": "text/event-stream", "Origin": BROWSER_ORIGIN,
            "ngrok-skip-browser-warning": "1"})
    saw_delta = False
    with urllib.request.urlopen(req, timeout=180) as response:
        if "text/event-stream" not in response.headers.get("Content-Type", ""):
            raise RuntimeError("公网没有返回 text/event-stream，请检查 ngrok 提示页或代理配置。")
        for raw in response:
            line = raw.decode("utf-8").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                if saw_delta:
                    return True
                raise RuntimeError("收到 [DONE]，但没有有效的 delta 数据。")
            event = json.loads(data)
            if "error" in event:
                raise RuntimeError("SSE 上游错误：" + str(event["error"]))
            saw_delta |= any("delta" in c for c in event.get("choices", []))
    raise RuntimeError("SSE 响应提前结束，未收到 [DONE]。")

# 本地 SSE 与公网 SSE 分开验证，区分网关和隧道问题。
try:
    check_stream(LOCAL_URL)
    print("本地 SSE 流式：验证通过", flush=True)
except Exception as e:
    raise RuntimeError("本地网关 SSE 自检失败：" + redact_ngrok(e)) from None

stream_ok = False
browser_ok = False
if public_ok:
    try:
        browser_ok = check_browser_access(url)
        print("公网浏览器跨域：验证通过", flush=True)
    except Exception as e:
        print("⚠️ 公网浏览器跨域检查未通过：", redact_ngrok(e), flush=True)
    try:
        stream_ok = check_stream(url)
    except Exception as e:
        print("⚠️ 公网 SSE 自检未通过：", redact_ngrok(e), flush=True)
else:
    print("⚠️ 公网模型列表未通过：", last_public_error, flush=True)
    print(redact_ngrok(public_diagnostic()), flush=True)
    print("若响应包含 ERR_NGROK_*，需按该错误码处理账号 / 域名 / 额度 / 连接问题。", flush=True)
    print("若 HTTP 401，请核对 API Key；若只有插件失败，请核对 OpenAI 兼容提供商和自定义请求头。", flush=True)

print("\n" + "=" * 60)
print("公网模型列表：", "验证通过" if public_ok else "尚未验证通过")
print("公网浏览器跨域：", "验证通过" if browser_ok else "尚未验证通过")
print("公网 SSE 流式：", "验证通过" if stream_ok else "尚未验证通过，请查看上方错误")
print("连接状态：", "可接入 Page Assist" if public_ok and browser_ok and stream_ok else "自检未全部通过，请先处理上方错误")
print("Base URL:", url + "/v1")
print("API Key :", API_KEY)
print("Model   :", MODEL_NAME)
print("模式    : ngrok HTTPS；支持 stream=true / false")
print("Page Assist 提供商：OpenAI Compatible / OpenAI 兼容（不要选择 Ollama 原生）")
print("Page Assist 自定义请求头：ngrok-skip-browser-warning = 1")
print("浏览器地址栏直接打开 API 不会自动携带 API Key，出现 401 属于正常鉴权。")
print("Notebook 会话停止后接口离线；域名通常仍归账号保留。")
print("持久化：Persistence = Files only 时，关机后保留 Ollama 安装包与模型，下次开机重新运行本单元即可。")
print("请勿公开含 API Key 的 Notebook 输出。日志目录:", ROOT)
print("=" * 60, flush=True)

def stop_services():
    """需要停止时，在另一个单元执行 stop_services()；仅停止本脚本的进程。"""
    config = globals().get("_ngrok_config")
    if config is not None:
        ngrok.kill(pyngrok_config=config)
    for name in ("tunnel_process", "gateway_process", "ollama_process"):
        process = globals().get(name)
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)
    print("本脚本启动的服务已停止。")

1/6 检查 GPU 与依赖...
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
2/6 安装 / 启动 Ollama...
复用持久化的 Ollama 安装包，正在解压...
Ollama {'version': '0.40.16'}
3/6 拉取指定模型的 Q4_K_M 量化版并创建模型名...
复用已下载的模型。


gathering model components gathering model components gathering model components 
using existing layer sha256:b4363b5710a29a03cab49bc10df92ca688f891eba1a63cae912e4d96f3fd7a65 
using existing layer sha256:f40af5231eaf9930131fc0c9247322211dac401cf3ae2f5232b958a3bf79c5a8 
using existing layer sha256:21b2f36f6a4fd8302b68ebd4fc81cd62e12acaa0d6034a24e2dd6817cd5ce643 
using existing layer sha256:37bb2a8e6154201237626d19e84649012c5d5530d11287e08b7d1f9b01f654d9 
using existing layer sha256:f4688a5b875eee0155cf21f32acfd16ac3e63804f32e0818ab1bca94063f5224 
writing manifest ⠋ 

模型能力： ['tools', 'thinking', 'completion', 'vision']
正在进行本地推理自检，首次加载需要等待...


gathering model components 
using existing layer sha256:b4363b5710a29a03cab49bc10df92ca688f891eba1a63cae912e4d96f3fd7a65 
using existing layer sha256:f40af5231eaf9930131fc0c9247322211dac401cf3ae2f5232b958a3bf79c5a8 
using existing layer sha256:21b2f36f6a4fd8302b68ebd4fc81cd62e12acaa0d6034a24e2dd6817cd5ce643 
using existing layer sha256:37bb2a8e6154201237626d19e84649012c5d5530d11287e08b7d1f9b01f654d9 
using existing layer sha256:f4688a5b875eee0155cf21f32acfd16ac3e63804f32e0818ab1bca94063f5224 
writing manifest 
success 


本地推理通过；GPU 权重字节： 5862210928
4/6 启动带 API Key 的 OpenAI 兼容网关...
本地网关：公网 Host / 扩展 Origin / OPTIONS 预检通过。
5/6 复用 / 创建 ngrok HTTPS 隧道...
6/6 验证公网模型列表和真实 SSE 响应...
本地 SSE 流式：验证通过
公网浏览器跨域：验证通过

公网模型列表： 验证通过
公网浏览器跨域： 验证通过
公网 SSE 流式： 验证通过
连接状态： 可接入 Page Assist
Base URL: https://359d-35-252-166-205.ngrok-free.app/v1
API Key : sk-_7aTajWxc6n1gbQFqT3Ojc7H-SZEH0UjB4vmR1B0Jcw
Model   : deepseek-v4-pro-qwen3.5-9b
模式    : ngrok HTTPS；支持 stream=true / false
Page Assist 提供商：OpenAI Compatible / OpenAI 兼容（不要选择 Ollama 原生）
Page Assist 自定义请求头：ngrok-skip-browser-warning = 1
浏览器地址栏直接打开 API 不会自动携带 API Key，出现 401 属于正常鉴权。
Notebook 会话停止后接口离线；域名通常仍归账号保留。
持久化：Persistence = Files only 时，关机后保留 Ollama 安装包与模型，下次开机重新运行本单元即可。
请勿公开含 API Key 的 Notebook 输出。日志目录: /kaggle/working/DeepSeek-Ollama
